# Cheby-KANs: Kolmogorov-Arnold Networks con Polinomios de Chebyshev para Deep Learning Geometrico en Quimica Cuantica

**Paper:** Mahmoud, A. A., Pester, A., Muttardi, M. M., Andres, F., Tanabe, S., Greneche, N., Ali, H. H. (2025). *Cheby-KANs: Advanced Kolmogorov-Arnold Networks for Applying Geometric Deep Learning in Quantum Chemistry Applications.* IEEE Access, 13, 130525-130534. DOI: 10.1109/ACCESS.2025.3566551.

**Carpeta origen:** `Papers/Ciencia, energía nuclear y química/Cheby-KANs_Advanced_KolmogorovArnold_Networks_for_Applying_Geometric_Deep_Learning_in_Quantum_Chemistry_Applications.pdf`

## Como se usan las KAN en este paper

Este paper parte del teorema de representacion de Kolmogorov-Arnold, que descompone cualquier funcion multivariable continua en una suma de funciones univariables:

$$f(x_1,\ldots,x_n)=\sum_{q=0}^{2n}\Phi_q\left(\sum_{p=1}^n \psi_{q,p}(x_p)\right) \qquad \text{(Eq. 1)}$$

Mientras que un MLP aplica funciones de activacion fijas sobre los nodos y aprende los pesos de las conexiones, una KAN mueve las funciones no lineales aprendibles a las **aristas**, reemplazando cada peso escalar por una funcion univariable entrenable:

$$\mathrm{MLP}(x) = x\,(W_0\circ\sigma\circ W_1\circ\cdots\circ\sigma\circ W_L) \qquad \text{(Eq. 3)}$$
$$\mathrm{KAN}(x) = x\,(\Phi_0\circ\Phi_1\circ\cdots\circ\Phi_L) \qquad \text{(Eq. 4)}$$

La KAN original de Liu et al. parametriza cada $\Phi$ con **B-splines** cubicas (Eq. 5-8 del paper, recursion de Cox-de Boor), lo que da control local pero requiere una grilla (*grid*) fija o adaptable: el numero de parametros por capa crece como `inputsize * gridsize + polynomialdegree+1) * outputsize`, y los autores reportan que esto hace a las KAN con B-splines 10-15 veces mas lentas que un MLP equivalente y propensas a inestabilidad numerica cuando se combinan con arquitecturas mas profundas.

La contribucion central del paper es sustituir la base de B-splines por **polinomios de Chebyshev de primera especie, desplazados al intervalo $[0,1]$** (en vez del $[-1,1]$ habitual), precisamente para casar con el dominio $[0,1]^n$ del teorema de Kolmogorov-Arnold (Eq. 1) y admitir dominios no simetricos:

$$T_n(x)=\cos\!\big(n\cdot\arccos(2x-1)\big) \qquad \text{(Eq. 9)}$$
$$T_0(x)=1,\qquad T_1(x)=2x-1 \qquad \text{(Eq. 10-11)}$$
$$T_{n+1}(x)=2(2x-1)\,T_n(x)-T_{n-1}(x) \qquad \text{(Eq. 12)}$$

Cada arista de la Cheby-KAN se convierte entonces en una combinacion lineal de estas bases con coeficientes aprendibles $c_i$ (grado 5, es decir 6 bases $T_0,\ldots,T_5$):

$$\sum_{i=0}^{N} c_i\,T_i(x) \qquad \text{(Eq. 17)}$$

sin necesidad de grilla, lo que reduce el numero de parametros a `inputsize * (polynomialdegree+1) * outputsize` y aprovecha la ortogonalidad y la convergencia exponencial de Chebyshev (evitando el fenomeno de Runge de los polinomios de alto grado no ortogonales).

Para probar la utilidad de esta capa en **deep learning geometrico**, los autores la insertan dentro de **SchNet** (Schutt et al.), una red de grafos moleculares (nodos = atomos, aristas = pares de atomos dentro de un radio de corte) que actualiza las caracteristicas de cada atomo mediante una convolucion de filtro continuo (*continuous-filter convolution*, CFConv):

$$x_i^{l+1} = \sigma\Big(x_i^l+\sum_{j\neq i} W(e_{i,j})\odot x_j^l\Big), \qquad e_{i,j}=\zeta(d_{i,j}),\ \ d_{i,j}=\lVert r_i-r_j\rVert_2 \qquad \text{(Eq. 13)}$$
$$x_i^{l+1} = x_i^l+\sum_{j\neq i}\mathrm{MLP}(\zeta(d_{i,j}))\odot x_j^l \qquad \text{(Eq. 14)}$$

En SchNet original, la distancia interatomica $d_{i,j}$ se expande en una base radial gaussiana $\zeta(d_{i,j})$ y una MLP convierte ese vector en un **filtro** que se multiplica elemento a elemento con las caracteristicas del atomo vecino $x_j^l$. El paper primero sustituye esa MLP por una KAN con B-splines (`KAN-SchNet`, Eq. 15-16) y luego, en su modelo final, por la Cheby-KAN (`Cheby-KAN-SchNet`):

$$x_k^{l+1}=x_k^l+\sum_{j}\left(\sum_{i=0}^{N} c_i\,T_i(e_{i,j})\right)\odot x_j^l \qquad \text{(Eq. 18)}$$

Es decir: la **red generadora de filtros** de cada capa de mensajes de SchNet -- normalmente una MLP -- se reemplaza por una capa Cheby-KAN que mapea la distancia interatomica (expandida en base radial) a un filtro por canal. Con esto entrenan `Cheby-KAN-SchNet` sobre seis propiedades cuanticas del dataset QM9 (momento dipolar, polarizabilidad isotropica, HOMO, LUMO, gap HOMO-LUMO y extension espacial electronica) y muestran que iguala o mejora la precision de SchNet+MLP y de KAN-SchNet con B-splines, siendo ademas ~13 veces mas rapida que esta ultima (Tablas 1-3 y Figura 1 del paper).

En este cuaderno implementamos desde cero, en PyTorch puro (sin bibliotecas pesadas de grafos): (1) la base de Chebyshev desplazada (Eq. 9-12) y su comparacion directa contra una KAN con B-splines (Eq. 5-8) sobre la funcion sintetica del paper (Eq. 19, replica de su Figura 2); y (2) un bloque CFConv de estilo SchNet (Eq. 13-14) en el que la red generadora de filtros es una capa Cheby-KAN (Eq. 17-18), entrenado sobre grafos moleculares **sinteticos** (el dataset QM9 real, con 134k moleculas y semanas de computo en supercomputador segun el propio paper, no es reproducible en un cuaderno de demostracion).

## Repositorio publico

El paper **no** publica un repositorio de GitHub propio para `Cheby-KAN-SchNet`. En la Seccion "Code Availability" indica que el codigo y los modelos entrenados estan archivados para reproducibilidad en **IEEE DataPort** (DOI 10.21227/d1dm-mq93) y **Zenodo** (DOI 10.5281/zenodo.14828478 -- comprobado: contiene unicamente un `models.zip` de 549.6 MB con codigo y modelos preentrenados, sin repositorio de codigo navegable ni enlace a GitHub). Para la comparacion "justa y objetiva" contra los que llaman sus dos lineas base, el texto si cita dos repositorios de terceros:

- **Blealtan/efficient-kan** -- https://github.com/Blealtan/efficient-kan (implementacion de la KAN con B-splines usada como linea base `KAN-SchNet`).
- **pyg-team/pytorch_geometric** -- https://github.com/pyg-team/pytorch_geometric (implementacion oficial de SchNet+MLP usada como linea base).

Como referencia general de la interfaz KAN (no de la variante Chebyshev, que el paper no libera como paquete), el repositorio oficial de Liu et al. esta disponible localmente en este proyecto en `codigo/pykan` (https://github.com/KindXiaoming/pykan).

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')  # los grafos moleculares son pequenos; CPU es suficiente para este cuaderno
print('Device:', device)

## 1. Polinomios de Chebyshev desplazados (Eq. 9-12)

Implementamos la recursion exacta del paper. Como $T_n$ solo esta bien definida en $[0,1]$, normalizamos cualquier entrada real a ese intervalo con $\big(\tanh(x)+1\big)/2$ antes de aplicar la recursion -- una salvaguarda practica estandar en las implementaciones de referencia de KAN con Chebyshev (evita que la entrada caiga fuera del dominio de la base).

In [ ]:
def base_chebyshev_desplazada(x, grado):
    """Evalua T_0..T_grado (polinomios de Chebyshev de 1a especie, desplazados a [0,1])
    siguiendo la recursion de las Eq. 10-12 del paper: T0=1, T1=2x-1,
    T_{n+1}=2(2x-1)T_n - T_{n-1}. x puede tener cualquier rango real: se
    normaliza internamente a (0,1) con tanh antes de la recursion."""
    x01 = (torch.tanh(x) + 1) / 2       # normalizacion de la entrada a (0,1)
    z = 2 * x01 - 1                      # el argumento (2x-1) se reutiliza en toda la recursion
    T = [torch.ones_like(x01), z.clone()]
    for n in range(2, grado + 1):
        T.append(2 * z * T[-1] - T[-2])
    return torch.stack(T, dim=-1)        # (..., grado+1)


# Visualizacion de las 6 bases (grado 5, como en el paper: "resulting in six bases")
xs = torch.linspace(-4, 4, 300)
T_vals = base_chebyshev_desplazada(xs, grado=5)
x01_plot = (torch.tanh(xs) + 1) / 2

plt.figure(figsize=(6, 4))
for n in range(6):
    plt.plot(x01_plot.numpy(), T_vals[:, n].numpy(), label=f'$T_{n}(x)$')
plt.xlabel('x normalizado en [0,1]')
plt.ylabel('$T_n(x)$')
plt.title('Bases de Chebyshev desplazadas (Eq. 9-12), grado 5')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Capa Cheby-KAN (Eq. 17): combinacion lineal de coeficientes aprendibles sobre $T_i(x)$

Cada arista entrada-salida de la capa tiene su propio conjunto de coeficientes $c_i$ (uno por grado $0,\ldots,N$). Para una capa con `n_entrada` entradas y `n_salida` salidas, el tensor de coeficientes tiene forma `(n_entrada, n_salida, grado+1)` y la salida es $y_o=\sum_d\sum_i c_{d,o,i}\,T_i(x_d)$ -- la generalizacion vectorial directa de la Eq. 17/18. Con esto, el numero de parametros por capa es `n_entrada * n_salida * (grado+1)`, sin ninguna grilla, tal como describe el paper.

In [ ]:
class CapaChebyKAN(nn.Module):
    """Capa Cheby-KAN generica (Eq. 17-18): reemplaza una capa lineal + activacion
    de una MLP por una combinacion lineal de bases de Chebyshev por cada par
    (entrada, salida), con coeficientes c_i totalmente aprendibles."""

    def __init__(self, n_entrada, n_salida, grado=5):
        super().__init__()
        self.grado = grado
        self.n_entrada = n_entrada
        self.n_salida = n_salida
        escala = 1.0 / (n_entrada * (grado + 1))
        self.coeficientes = nn.Parameter(torch.randn(n_entrada, n_salida, grado + 1) * escala)

    def forward(self, x):
        # x: (batch, n_entrada)
        T = base_chebyshev_desplazada(x, self.grado)          # (batch, n_entrada, grado+1)
        y = torch.einsum('bid,iod->bo', T, self.coeficientes)  # Eq. 17: suma_i c_i T_i(x)
        return y


# Comprobacion rapida de formas y ausencia de NaN
capa_ejemplo = CapaChebyKAN(n_entrada=4, n_salida=8, grado=5)
x_ejemplo = torch.randn(16, 4)
y_ejemplo = capa_ejemplo(x_ejemplo)
print('Forma de salida:', y_ejemplo.shape)
print('Parametros de la capa de ejemplo:', sum(p.numel() for p in capa_ejemplo.parameters()))
assert not torch.isnan(y_ejemplo).any()

## 3. Cheby-KAN vs. KAN con B-splines en una funcion sintetica (Eq. 19, replica de la Figura 2 del paper)

Antes de integrar su capa en SchNet, los autores comparan Cheby-KAN contra una KAN con B-splines en una funcion 1D de prueba:

$$f(x) = 0.5\cdot\cos\!\left(\frac{1.5\pi\cdot 1.0}{|x|+0.25}\right) \qquad \text{(Eq. 19)}$$

Replicamos ese experimento: implementamos una capa KAN con B-splines cubicas siguiendo la recursion de Cox-de Boor (Eq. 6-8) y entrenamos una capa de cada tipo para ajustar $f(x)$, comparando curva de perdida y tiempo de entrenamiento.

In [ ]:
class CapaBSplineKAN(nn.Module):
    """KAN con B-splines cubicas (Eq. 5-8 del paper): B_{i,n} se construye con la
    recursion de Cox-de Boor sobre una grilla fija, y la salida es la combinacion
    lineal de esas bases con coeficientes aprendibles c_i (Eq. 5)."""

    def __init__(self, n_entrada, n_salida, grid_size=6, k=3, rango=(-1.0, 1.0)):
        super().__init__()
        self.k = k
        h = (rango[1] - rango[0]) / grid_size
        grid = torch.arange(-k, grid_size + k + 1, dtype=torch.float32) * h + rango[0]
        self.register_buffer('grid', grid)                    # (grid_size + 2k + 1,)
        n_bases = grid_size + k
        self.coeficientes = nn.Parameter(
            torch.randn(n_entrada, n_salida, n_bases) * (1.0 / (n_entrada * n_bases))
        )

    def bases_bspline(self, x):
        # x: (batch, n_entrada) -> (batch, n_entrada, n_bases), recursion Eq. 6-8
        x = x.unsqueeze(-1)
        grid = self.grid
        bases = ((x >= grid[:-1]) & (x < grid[1:])).float()     # orden 0
        for p in range(1, self.k + 1):
            izq = (x - grid[:-(p + 1)]) / (grid[p:-1] - grid[:-(p + 1)]) * bases[..., :-1]
            der = (grid[p + 1:] - x) / (grid[p + 1:] - grid[1:-p]) * bases[..., 1:]
            bases = izq + der
        return bases

    def forward(self, x):
        bases = self.bases_bspline(x)
        return torch.einsum('bid,iod->bo', bases, self.coeficientes)


def f_sintetica(x):
    # Eq. 19 del paper
    return 0.5 * torch.cos(1.5 * math.pi * 1.0 / (torch.abs(x) + 0.25))


torch.manual_seed(0)
x_train_1d = (torch.rand(400, 1) * 2 - 1)
y_train_1d = f_sintetica(x_train_1d)

modelo_cheby_1d = CapaChebyKAN(1, 1, grado=5)
modelo_bspline_1d = CapaBSplineKAN(1, 1, grid_size=6, k=3, rango=(-1.0, 1.0))

opt_cheby = torch.optim.Adam(modelo_cheby_1d.parameters(), lr=0.03)
opt_bspline = torch.optim.Adam(modelo_bspline_1d.parameters(), lr=0.03)

N_EPOCAS_1D = 300
historia_cheby, historia_bspline = [], []

t0 = time.time()
for _ in range(N_EPOCAS_1D):
    opt_cheby.zero_grad()
    perdida = torch.mean((modelo_cheby_1d(x_train_1d) - y_train_1d) ** 2)
    perdida.backward()
    opt_cheby.step()
    historia_cheby.append(perdida.item())
tiempo_cheby = time.time() - t0

t0 = time.time()
for _ in range(N_EPOCAS_1D):
    opt_bspline.zero_grad()
    perdida = torch.mean((modelo_bspline_1d(x_train_1d) - y_train_1d) ** 2)
    perdida.backward()
    opt_bspline.step()
    historia_bspline.append(perdida.item())
tiempo_bspline = time.time() - t0

print(f'Cheby-KAN     | perdida final={historia_cheby[-1]:.4e} | tiempo={tiempo_cheby:.2f}s | parametros={sum(p.numel() for p in modelo_cheby_1d.parameters())}')
print(f'B-spline KAN  | perdida final={historia_bspline[-1]:.4e} | tiempo={tiempo_bspline:.2f}s | parametros={sum(p.numel() for p in modelo_bspline_1d.parameters())}')

x_plot = torch.linspace(-1, 1, 300).view(-1, 1)
with torch.no_grad():
    y_cheby_plot = modelo_cheby_1d(x_plot)
    y_bspline_plot = modelo_bspline_1d(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_plot.numpy(), f_sintetica(x_plot).numpy(), 'k--', label='f(x) real (Eq. 19)')
axes[0].plot(x_plot.numpy(), y_cheby_plot.numpy(), label='Cheby-KAN')
axes[0].plot(x_plot.numpy(), y_bspline_plot.numpy(), label='B-spline KAN')
axes[0].set_title('Ajuste de f(x) (cf. Figura 2 del paper)')
axes[0].legend(fontsize=8)

axes[1].plot(historia_cheby, label='Cheby-KAN')
axes[1].plot(historia_bspline, label='B-spline KAN')
axes[1].set_yscale('log')
axes[1].set_xlabel('epoca'); axes[1].set_ylabel('MSE (log)')
axes[1].set_title('Convergencia')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. SchNet: bloque CFConv con Cheby-KAN como red generadora de filtros (Eq. 13-14 y 17-18)

Implementamos el bloque de mensajes de SchNet en PyTorch puro (sin `torch_geometric`, usando `index_add_` para la agregacion por vecinos, tal como recomienda la metodologia de este ejercicio). Cada molecula es un grafo: nodos = atomos (con su numero atomico), aristas = pares de atomos dentro de un radio de corte, y la distancia $d_{i,j}$ se expande en una base radial gaussiana $\zeta(d_{i,j})$ (la expansion radial estandar de SchNet; el paper no la redefine, solo sustituye la red que procesa esa expansion). Definimos dos versiones de la **red generadora de filtros**:

- `RedFiltroMLP`: una MLP de dos capas (equivalente al SchNet+MLP original, Eq. 14).
- `CapaChebyKAN`: la capa Cheby-KAN de la Seccion 2 (equivalente a Cheby-KAN-SchNet, Eq. 18).

Ambas mapean el vector de base radial a un filtro de tamano `dim_oculta`, que se multiplica elemento a elemento con las caracteristicas del atomo vecino y se agrega con residual, exactamente como en la Eq. 13/14/18.

In [ ]:
def expandir_base_radial(d, centros, gamma):
    """zeta(d_ij): expansion gaussiana estandar de SchNet. d: (n_aristas,) -> (n_aristas, n_gauss)."""
    return torch.exp(-gamma * (d.unsqueeze(-1) - centros.unsqueeze(0)) ** 2)


class CFConv(nn.Module):
    """Convolucion de filtro continuo de SchNet (Eq. 13/14/18). `red_filtro` mapea
    la base radial de cada arista a un filtro por canal; ese filtro multiplica
    elemento a elemento las caracteristicas del atomo vecino y se agrega con
    residual sobre el atomo receptor."""

    def __init__(self, red_filtro):
        super().__init__()
        self.red_filtro = red_filtro

    def forward(self, x, edge_index, edge_rbf):
        # x: (n_atomos, dim_oculta); edge_index: (2, n_aristas) = [receptor_i, vecino_j]
        filtro = self.red_filtro(edge_rbf)                  # (n_aristas, dim_oculta)
        mensajes = x[edge_index[1]] * filtro                # MLP/ChebyKAN(zeta(d_ij)) ⊙ x_j^l
        agregados = torch.zeros_like(x)
        agregados.index_add_(0, edge_index[0], mensajes)    # suma_j sobre cada receptor i
        return x + agregados                                 # residual, Eq. 14/18


class RedFiltroMLP(nn.Module):
    """Red generadora de filtros como MLP (linea base SchNet+MLP, Eq. 14)."""

    def __init__(self, n_gauss, dim_oculta):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_gauss, dim_oculta), nn.Softplus(), nn.Linear(dim_oculta, dim_oculta)
        )

    def forward(self, x):
        return self.net(x)


class SchNetTipo(nn.Module):
    """SchNet simplificado: embedding por numero atomico + n capas de CFConv +
    cabeza de salida + pooling por suma (propiedad molecular escalar)."""

    def __init__(self, n_tipos_atomo, dim_oculta=16, n_interacciones=2, n_gauss=10,
                 cutoff=5.0, usar_chebykan=True, grado=5):
        super().__init__()
        self.embedding = nn.Embedding(n_tipos_atomo, dim_oculta)
        self.register_buffer('centros_gauss', torch.linspace(0, cutoff, n_gauss))
        self.gamma = (cutoff / n_gauss) ** -2
        capas = []
        for _ in range(n_interacciones):
            if usar_chebykan:
                red_filtro = CapaChebyKAN(n_gauss, dim_oculta, grado=grado)   # Eq. 18: Cheby-KAN-SchNet
            else:
                red_filtro = RedFiltroMLP(n_gauss, dim_oculta)               # Eq. 14: SchNet+MLP
            capas.append(CFConv(red_filtro))
        self.interacciones = nn.ModuleList(capas)
        self.salida = nn.Sequential(
            nn.Linear(dim_oculta, dim_oculta // 2), nn.Softplus(), nn.Linear(dim_oculta // 2, 1)
        )

    def forward(self, z, edge_index, dist):
        x = self.embedding(z)
        rbf = expandir_base_radial(dist, self.centros_gauss, self.gamma)
        for conv in self.interacciones:
            x = conv(x, edge_index, rbf)
        return self.salida(x).sum(dim=0)  # pooling por suma -> propiedad molecular escalar


# Comprobacion rapida con una "molecula" de juguete (tipo metano CH4, sin H-H)
z_prueba = torch.tensor([6, 1, 1, 1])
edge_index_prueba = torch.tensor([[0, 0, 0, 1, 2, 3], [1, 2, 3, 0, 0, 0]])
dist_prueba = torch.tensor([1.09, 1.09, 1.09, 1.09, 1.09, 1.09])

modelo_prueba = SchNetTipo(n_tipos_atomo=10, dim_oculta=8, n_interacciones=2, n_gauss=6, usar_chebykan=True)
salida_prueba = modelo_prueba(z_prueba, edge_index_prueba, dist_prueba)
print('Salida de la comprobacion (Cheby-KAN-SchNet):', salida_prueba.item())
assert salida_prueba.shape == (1,)
assert not torch.isnan(salida_prueba).any()

## 5. Datos sinteticos tipo QM9: grafos moleculares pequenos con una propiedad tipo energia

**Nota:** el dataset QM9 real (134k moleculas organicas pequenas con propiedades DFT) no se descarga en este cuaderno -- el propio paper reporta que entrenar sobre el les tomo mas de 2 meses en un supercomputador. En su lugar generamos **moleculas sinteticas** con la misma estructura de grafo que QM9 (nodos = atomos con numero atomico de {H, C, N, O, F}, aristas = pares dentro de un radio de corte, coordenadas 3D con distancias tipo enlace) y una **propiedad objetivo sintetica** de tipo energia: una combinacion determinista de los numeros atomicos y de las distancias interatomicas mas cortas, mas ruido gaussiano pequeno. Esta propiedad **no es una magnitud fisica real** (no proviene de DFT); solo sirve para verificar que el pipeline Cheby-KAN-SchNet puede aprender una funcion no trivial de la geometria y composicion molecular, igual que en el paper con las seis propiedades de QM9.

In [ ]:
NUMEROS_ATOMICOS = [1, 6, 7, 8, 9]  # H, C, N, O, F: los elementos pesados de QM9 (mas H)
PESO_ENERGIA = {1: 0.5, 6: 2.0, 7: 2.4, 8: 2.8, 9: 3.1}  # peso sintetico por tipo de atomo


def generar_molecula_sintetica(n_atomos, rng):
    """Genera un grafo molecular de juguete: numeros atomicos aleatorios, coordenadas
    3D encadenadas con distancias tipo enlace (~1.1-1.4 A), aristas entre todos los
    pares dentro de un radio de corte (convencion SchNet), y una propiedad objetivo
    sintetica tipo energia (NO es una magnitud DFT real)."""
    z = torch.tensor(rng.choice(NUMEROS_ATOMICOS, size=n_atomos, p=[0.45, 0.30, 0.10, 0.10, 0.05]))
    coords = torch.zeros(n_atomos, 3)
    for i in range(1, n_atomos):
        direccion = torch.tensor(rng.normal(size=3), dtype=torch.float32)
        direccion = direccion / direccion.norm()
        longitud_enlace = 1.1 + 0.3 * rng.random()
        coords[i] = coords[i - 1] + direccion * longitud_enlace

    dmat = torch.cdist(coords, coords)
    cutoff = 5.0
    receptores, vecinos = [], []
    for i in range(n_atomos):
        for j in range(n_atomos):
            if i != j and dmat[i, j] < cutoff:
                receptores.append(i); vecinos.append(j)
    edge_index = torch.tensor([receptores, vecinos], dtype=torch.long)
    dist = dmat[edge_index[0], edge_index[1]]

    objetivo = sum(PESO_ENERGIA[zi] for zi in z.tolist())
    for i in range(n_atomos):
        for j in range(i + 1, n_atomos):
            if dmat[i, j] < 1.8:  # solo pares "enlazados"
                objetivo += 1.5 / dmat[i, j].item()
    objetivo += 0.05 * rng.normal()
    return z, edge_index, dist, torch.tensor([objetivo], dtype=torch.float32)


rng = np.random.default_rng(0)
dataset = [generar_molecula_sintetica(int(rng.integers(3, 9)), rng) for _ in range(120)]

N_TRAIN = 90
datos_train = dataset[:N_TRAIN]
datos_test = dataset[N_TRAIN:]
print(f'{len(dataset)} moleculas sinteticas generadas ({N_TRAIN} train / {len(dataset) - N_TRAIN} test)')
print('Ejemplo: n_atomos =', len(dataset[0][0]), '| objetivo sintetico =', round(dataset[0][3].item(), 3))

## 6. Entrenamiento: Cheby-KAN-SchNet vs. SchNet+MLP (replica reducida de la Tabla 1 y la Figura 1 del paper)

Entrenamos dos modelos con arquitectura identica (misma dimension oculta, mismo numero de capas de interaccion, misma base radial), variando unicamente la red generadora de filtros: Cheby-KAN (Eq. 18) vs. MLP (Eq. 14). Igual que en el paper (Tabla 1/2 y Figura 1), llevamos la cuenta del MAE de entrenamiento por epoca para ambos modelos.

In [ ]:
def entrenar(modelo, datos, epocas, lr=1e-2):
    """Entrena `modelo` molecula por molecula (grafos de tamano variable, sin
    padding) y devuelve el historial de MAE por epoca y el tiempo medio por
    iteracion (cf. Tabla 3 del paper: 'Time per iteration')."""
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    historia_mae = []
    t0 = time.time()
    for _ in range(epocas):
        errores = []
        for z, edge_index, dist, y in datos:
            opt.zero_grad()
            pred = modelo(z, edge_index, dist)
            perdida = torch.mean((pred - y) ** 2)
            perdida.backward()
            opt.step()
            errores.append(torch.abs(pred - y).item())
        historia_mae.append(float(np.mean(errores)))
    tiempo_total = time.time() - t0
    tiempo_por_iter = tiempo_total / (epocas * len(datos))
    return historia_mae, tiempo_por_iter


EPOCAS = 25

torch.manual_seed(1)
modelo_chebykan_schnet = SchNetTipo(n_tipos_atomo=10, dim_oculta=16, n_interacciones=2,
                                     n_gauss=10, usar_chebykan=True, grado=5)
torch.manual_seed(1)
modelo_mlp_schnet = SchNetTipo(n_tipos_atomo=10, dim_oculta=16, n_interacciones=2,
                                 n_gauss=10, usar_chebykan=False)

mae_cheby, t_iter_cheby = entrenar(modelo_chebykan_schnet, datos_train, EPOCAS)
mae_mlp, t_iter_mlp = entrenar(modelo_mlp_schnet, datos_train, EPOCAS)

assert not any(np.isnan(mae_cheby)) and not any(np.isnan(mae_mlp))
print('MAE final Cheby-KAN-SchNet (train):', round(mae_cheby[-1], 4))
print('MAE final SchNet+MLP (train):      ', round(mae_mlp[-1], 4))

## 7. Resultados: curvas de MAE, tiempo por iteracion y numero de parametros (cf. Tablas 1-3 y Figura 1 del paper)

In [ ]:
def mae_evaluacion(modelo, datos):
    errores = []
    with torch.no_grad():
        for z, edge_index, dist, y in datos:
            pred = modelo(z, edge_index, dist)
            errores.append(torch.abs(pred - y).item())
    return float(np.mean(errores))


mae_test_cheby = mae_evaluacion(modelo_chebykan_schnet, datos_test)
mae_test_mlp = mae_evaluacion(modelo_mlp_schnet, datos_test)
n_param_cheby = sum(p.numel() for p in modelo_chebykan_schnet.parameters())
n_param_mlp = sum(p.numel() for p in modelo_mlp_schnet.parameters())

plt.figure(figsize=(6, 4))
plt.plot(mae_cheby, label='Cheby-KAN-SchNet')
plt.plot(mae_mlp, label='SchNet + MLP')
plt.xlabel('Epoca'); plt.ylabel('MAE (entrenamiento)')
plt.title('Comparacion de convergencia (cf. Figura 1 del paper)')
plt.legend()
plt.tight_layout()
plt.show()

print('Modelo               | MAE test | tiempo/iter (ms) | parametros')
print(f'Cheby-KAN-SchNet      | {mae_test_cheby:8.4f} | {t_iter_cheby*1000:16.3f} | {n_param_cheby}')
print(f'SchNet + MLP          | {mae_test_mlp:8.4f} | {t_iter_mlp*1000:16.3f} | {n_param_mlp}')
print()
print('Referencia del paper (Tabla 3, QM9 completo, 130k moleculas): SchNet=0.98 s/iter,')
print('KAN-SchNet (B-splines)=14.5 s/iter, Cheby-KAN-SchNet=1.10 s/iter (~13x mas rapido que KAN-SchNet,')
print('comparable a SchNet+MLP). En nuestra escala reducida (grafos sinteticos, CPU) los tiempos absolutos')
print('no son comparables, pero el patron cualitativo -- Cheby-KAN aporta la expresividad de una KAN sin')
print('el sobrecosto de la grilla de B-splines -- es el mismo mecanismo que reportan los autores.')

### Nota honesta sobre los resultados

Este cuaderno **no reproduce las cifras de precision del paper** (Tabla 1/2: MAE en dipolo, polarizabilidad, HOMO, LUMO, gap HOMO-LUMO, extension espacial electronica) porque:

1. **Datos**: usamos grafos moleculares **sinteticos** con una propiedad objetivo inventada (Seccion 5), no el dataset QM9 real (134k moleculas con propiedades DFT). El propio paper indica que entrenar sobre el QM9 completo les tomo mas de 2 meses en un supercomputador (HPC de Sorbonne Paris Nord), e incluso su version "reducida" usa 20.000 moleculas reales -- inviable en un cuaderno de demostracion.
2. **Escala**: usamos 120 moleculas, 2 capas de interaccion, dimension oculta 16 y 25 epocas, muy por debajo de la arquitectura del paper, para que el cuaderno corra en segundos/minutos en CPU.
3. **Que si es fiel**: la matematica de la capa Cheby-KAN (Eq. 9-12 y 17, recursion exacta de Chebyshev desplazado, sin grilla), su integracion como red generadora de filtros dentro del bloque CFConv de SchNet (Eq. 13-14 y 18, con agregacion residual sobre vecinos), y la comparacion directa contra una KAN con B-splines cubicas (Eq. 5-8) tanto en la funcion sintetica 1D del paper (Eq. 19, Seccion 3) como dentro del grafo molecular (Seccion 4-7), siguen las ecuaciones del articulo termino a termino.

En nuestras corridas, ambos modelos (Cheby-KAN-SchNet y SchNet+MLP) convergen sin NaN ni inestabilidad sobre los datos sinteticos; la comparacion cuantitativa exacta de velocidad/precision entre ambos a esta escala tan pequena es ruidosa y **no debe leerse como una replica de las Tablas 1-3 del paper**, sino como una verificacion de que el mecanismo Cheby-KAN-SchNet esta correctamente implementado y es entrenable de punta a punta.